# Advanced Reinforcement Learning Project - Experiment 1

In [4]:
!pip install gymnasium[box2d] swig renderlab torch matplotlib moviepy==1.0.3 opencv-python

   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
    --------------------------------------- 0.8/40.2 MB 6.7 MB/s eta 0:00:06
   --- ------------------------------------ 3.1/40.2 MB 10.3 MB/s eta 0:00:04
   ----- ---------------------------------- 5.8/40.2 MB 10.7 MB/s eta 0:00:04
   -------- ------------------------------- 8.1/40.2 MB 11.0 MB/s eta 0:00:03
   -------- ------------------------------- 8.7/40.2 MB 9.8 MB/s eta 0:00:04
   ---------- ----------------------------- 10.5/40.2 MB 9.0 MB/s eta 0:00:04
   ------------ --------------------------- 12.3/40.2 MB 9.0 MB/s eta 0:00:04
   -------------- ------------------------- 14.9/40.2 MB 9.3 MB/s eta 0:00:03
   ---------------- ----------------------- 17.0/40.2 MB 9.3 MB/s eta 0:00:03
   ------------------- -------------------- 19.4/40.2 MB 9.5 MB/s eta 0:00:03
   --------------------- ------------------ 21.5/40.2 MB 9.6 MB/s eta 0:00:02
   ----------------------- ---------------- 23.6/40.2 MB 9.6 MB/s eta 0:00

In [5]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from collections import namedtuple, deque
import itertools
import random
import warnings
import renderlab as rl
warnings.filterwarnings("ignore")

# Configures device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


## Hyperparameters

In [6]:
# Defining default hyperparameters suited for LunarLander-v3
GAMMA = 0.99
BATCH_SIZE = 128
BUFFER_SIZE = 100000
MIN_REPLAY_SIZE = 1000
EPS_START = 0.9
EPS_END = 0.05
EPS_DECAY = 0.995
TARGET_UPDATE_FREQ = 10
LR = 5e-4

## Replay Memory and DQN Class

In [ ]:
Transition = namedtuple('Transition', ('states', 'actions', 'rewards', 'dones', 'next_states'))

class Replay_memory():
    def __init__(self, env, fullsize, minsize, batchsize, seed=None):
        self.env = env
        self.memory = deque(maxlen=fullsize)
        self.rewards = deque(maxlen=50)
        self.batchsize = batchsize
        self.minsize = minsize

    def append(self, transition):
        self.memory.append(transition)

    def sample_batch(self):
        batch = random.sample(self.memory, self.batchsize)
        batch = Transition(*zip(*batch))
        states = torch.from_numpy(np.array(batch.states, dtype=np.float32)).to(device)
        actions = torch.from_numpy(np.array(batch.actions, dtype=np.int64)).unsqueeze(1).to(device)
        rewards = torch.from_numpy(np.array(batch.rewards, dtype=np.float32)).unsqueeze(1).to(device)
        dones = torch.from_numpy(np.array(batch.dones, dtype=bool)).unsqueeze(1).to(device)
        next_states = torch.from_numpy(np.array(batch.next_states, dtype=np.float32)).to(device)
        return states, actions, rewards, dones, next_states

    def initialize(self):
        obs, _ = self.env.reset(seed=self.seed)
        for _ in range(self.minsize):
            action = self.env.action_space.sample()
            new_obs, reward, terminated, truncated, _ = self.env.step(action)
            done = terminated or truncated
            transition = Transition(obs, action, reward, done, new_obs)
            self.append(transition)
            obs = new_obs
            if done:
                self.env.reset()
        return self

In [8]:
class DQN(nn.Module):
    def __init__(self, ninputs, noutputs, hidden_size=64):
        super(DQN, self).__init__()
        # Modified to use ReLU and 2 hidden layers due to LunarLander complexity
        self.a1 = nn.Linear(ninputs, hidden_size)
        self.a2 = nn.Linear(hidden_size, hidden_size)
        self.a3 = nn.Linear(hidden_size, noutputs)

    def forward(self, X):
        o = self.a1(X)
        o = F.relu(o)
        o = self.a2(o)
        o = F.relu(o)
        o = self.a3(o)
        return o

    def __call__(self, X):
        return self.forward(X)

In [9]:
def epsilon_greedy_policy(policy_net, epsilon, obs, env):
    rnd_sample = random.random()
    if rnd_sample <= epsilon:
        action = env.action_space.sample()
    else:
        with torch.no_grad():
            # obs needs to be tensor on device
            obs_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)
            action = int(torch.argmax(policy_net(obs_t)))
    return action

## Training Logic

In [ ]:
# Function used to run training
def run_training(hyperparams_dict, name="DQN_Run", double_dqn=False, num_episodes=1000, seed=50):
    # Unpacks hyperparameters
    gamma = hyperparams_dict.get('GAMMA', GAMMA)
    batch_size = hyperparams_dict.get('BATCH_SIZE', BATCH_SIZE)
    buffer_size = hyperparams_dict.get('BUFFER_SIZE', BUFFER_SIZE)
    min_replay_size = hyperparams_dict.get('MIN_REPLAY_SIZE', MIN_REPLAY_SIZE)
    eps_start = hyperparams_dict.get('EPS_START', EPS_START)
    eps_end = hyperparams_dict.get('EPS_END', EPS_END)
    eps_decay = hyperparams_dict.get('EPS_DECAY', EPS_DECAY)
    tqf = hyperparams_dict.get('TQF', TARGET_UPDATE_FREQ)
    lr = hyperparams_dict.get('LR', LR)
    
    # Initialises seeds for reproducibility
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    # Creates an environment with no rendering since it is not needed for now
    env = gym.make("LunarLander-v3", render_mode=None)
    replay_memory = Replay_memory(env, buffer_size, min_replay_size, batch_size, seed=seed).initialize()
    
    dqn_policy = DQN(env.observation_space.shape[0], env.action_space.n).to(device)
    dqn_target = DQN(env.observation_space.shape[0], env.action_space.n).to(device)
    dqn_target.load_state_dict(dqn_policy.state_dict())
    dqn_target.eval()

    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(dqn_policy.parameters(), lr=lr)
    

    obs, _ = env.reset()
    eps_threshold = eps_start
    episode = 1
    episode_reward = 0
    
    # Used to store rewards history
    rewards_history = []

    # Prints training info stating which DQN is being used and hyperparameters being evaluated
    if double_dqn:
        print(f"[{name}] Starting... DoubleDQN | LR={lr} | BS={batch_size} | TQF={tqf}")
    else:
        print(f"[{name}] Starting... DQN | LR={lr} | BS={batch_size} | TQF={tqf}")

    for step in itertools.count():
        action = epsilon_greedy_policy(dqn_policy, eps_threshold, obs, env)
        new_obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        replay_memory.append(Transition(obs, action, reward, done, new_obs))
        episode_reward += reward
        obs = new_obs

        if done:
            replay_memory.rewards.append(episode_reward)
            rewards_history.append(episode_reward)
            
            eps_threshold = max(eps_end, eps_threshold * eps_decay)
            
            obs, _ = env.reset()
            episode_reward = 0
            
            # Checks solved condition
            avg_res = np.mean(list(replay_memory.rewards)) # rewards deque has maxlen=50
            
            if episode % 50 == 0:
                print(f'Episode: {episode} Avg Results: {avg_res:.2f} Epsilon: {eps_threshold:.2f}')
            
            if avg_res >= 195.0:
                print(f'Solved at episode: {episode} Avg Results: {avg_res:.2f}')
                break
            
            episode += 1
            if episode > num_episodes:
                break

        # Trains step
        b_states, b_actions, b_rewards, b_dones, b_next_states = replay_memory.sample_batch()

        qvalues = dqn_policy(b_states).gather(1, b_actions)

        with torch.no_grad():
            # Checks for double dqn and uses the policy network to select the next action
            if double_dqn:
                next_actions = dqn_policy(b_next_states).argmax(dim=1, keepdim=True)
                next_qvalues = dqn_target(b_next_states).gather(1, next_actions)
            else:
                next_qvalues = dqn_target(b_next_states).max(1, keepdim=True)[0]
                
            expected_qvalues = b_rewards + gamma * (1 - b_dones.type(torch.int64)) * next_qvalues

        loss = loss_fn(qvalues, expected_qvalues)
        optimizer.zero_grad()
        loss.backward()
        for param in dqn_policy.parameters():
            param.grad.data.clamp_(-1, 1)
        optimizer.step()

        if episode % tqf == 0:
            dqn_target.load_state_dict(dqn_policy.state_dict())
            
    # Closes environment and returns rewards history and final policy after training
    env.close()
    return rewards_history, dqn_policy

## Training and Hyperparameter Tuning

In [12]:
def tune_hyperparameters(double_dqn):
    # Grid Search used to find best hyperparameters
    BATCH_SIZES = [64, 128, 256]
    LRS = [5e-4, 1e-3]
    TQFs = [10, 100]
    grid_results = {}
    # Initialises best average reward to negative infinity and improves as the code runs
    best_avg_reward = -float('inf')
    best_params = {}
    best_model = None
    best_scores = None

    print("Starting Grid Search...")

    # Loops through all combinations of hyperparameters
    for bs in BATCH_SIZES:
        for lr in LRS:
            for tqf in TQFs:
                current_params = {
                    'BATCH_SIZE': bs,
                    'LR': lr,
                    'TQF': tqf
                }
                algo = "DDQN" if double_dqn else "DQN"
                run_name = f"{algo}_BS{bs}_LR{lr}_TQF{tqf}"
                print(f"\n--- Running {run_name} ---")
                # Runs training with current parameters
                scores, model = run_training(current_params, name=run_name, double_dqn=double_dqn)
                
                # Stores results
                grid_results[run_name] = scores
                
                # Checks if best (last 50 ep avg)
                avg_reward = np.mean(scores[-50:]) if len(scores) >= 50 else np.mean(scores)
                if avg_reward > best_avg_reward:
                    best_avg_reward = avg_reward
                    best_params = current_params
                    best_model = model
                    best_scores = scores
                    print(f"New Best: {run_name} (Avg: {avg_reward:.2f})")

    print(f"\nGrid Search Complete. Best Params: {best_params}")
    return best_params, best_model, best_scores, grid_results

# Runs grid search for both DQN and DDQN and gets the best models, scores and grid results   
DQN_params, DQN_model, DQN_scores, DQN_grid_results = tune_hyperparameters(double_dqn=False)
DDQN_params, DDQN_model, DDQN_scores, DDQN_grid_results = tune_hyperparameters(double_dqn=True)

Starting Grid Search...

--- Running DQN_BS64_LR0.0005_TQF10 ---
[DQN_BS64_LR0.0005_TQF10] Starting... DQN | LR=0.0005 | BS=64 | TQF=10


KeyboardInterrupt: 

## Comparisons

In [ ]:
# Function used to plot the grid search results
def grid_plot_comparison(grid_results, title):
    plt.figure(figsize=(12, 8))
    
    # Plots all grid search results
    for name, scores in grid_results.items():
        window = 50
        if len(scores) >= window:
            moving_avg = np.convolve(scores, np.ones(window)/window, mode='valid')
            x = np.arange(window-1, len(scores))  # Corrects x-axis offset
            plt.plot(x, moving_avg, label=name)
        else:
            plt.plot(scores, label=name)

    plt.xlabel('Episode')
    plt.ylabel('Average Reward (50-ep)')
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

# Function used to plot the best models
def plot_comparison(scores_1, scores_2, title):
    plt.figure(figsize=(12, 8))
            
    # Plots DQN
    dqn_label = "DQN (Best)"
    if len(scores_1) >= 50:
        ma_dqn = np.convolve(scores_1, np.ones(50)/50, mode='valid')
        x_dqn = np.arange(49, len(scores_1)) 
        plt.plot(x_dqn, ma_dqn, label=dqn_label, linewidth=3, linestyle='--', color='blue')
    else:
        plt.plot(scores_1, label=dqn_label, linewidth=3, linestyle='--', color='blue')
    
    # Plots DDQN
    ddqn_label = "Double DQN (Best)"
    if len(scores_2) >= 50:
        ma_ddqn = np.convolve(scores_2, np.ones(50)/50, mode='valid')
        x_ddqn = np.arange(49, len(scores_2)) 
        plt.plot(x_ddqn, ma_ddqn, label=ddqn_label, linewidth=3, linestyle='--', color='red')
    else:
        plt.plot(scores_2, label=ddqn_label, linewidth=3, linestyle='--', color='red')

    plt.xlabel('Episode')
    plt.ylabel('Average Reward (50-ep)')
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

grid_plot_comparison(DQN_grid_results, 'Grid Search of DQN')
grid_plot_comparison(DDQN_grid_results, 'Grid Search of Double DQN')
plot_comparison(DQN_scores, DDQN_scores, 'Best Models Comparison')

## Analysis & Robustness
Checks performance under wind and turbulence.

In [ ]:
# Function used to test robustness DQNs against wind and turbulence
def test_robustness(agent, wind_powers, turbulence_powers, num_episodes, seed=50):
    # Initialises a 2D array to store results for each combination
    results = np.zeros((len(wind_powers), len(turbulence_powers)))
    
    # Loops through each combination of wind and turbulence
    for i, w in enumerate(wind_powers):
        for j, t in enumerate(turbulence_powers):
            # Environments with wind and turbulence
            env_test = gym.make("LunarLander-v3", enable_wind=True, wind_power=w, turbulence_power=t)
            total = 0
            for ep in range(num_episodes):
                obs, _ = env_test.reset(seed=seed if ep == 0 else None)
                done = False
                while not done:
                    # 0.0 due to no exploration and tests the agent's learned behavior
                    action = epsilon_greedy_policy(agent, 0.0, obs, env_test)
                    obs, r, term, trunc, _ = env_test.step(action)
                    done = term or trunc
                    total += r
            
            results[i, j] = total / num_episodes
            env_test.close()
    return results

# Defines wind and turbulence parameters
winds = [0.0, 5.0, 10.0, 15.0, 20.0]
turbulences = [0.0, 0.5, 1.0, 1.5, 2.0]
num_test_eps = 50

print("Testing Robustness (Standard DQN)...")
res_std = test_robustness(DQN_model, winds, turbulences, num_test_eps)

print("Testing Robustness (Double DQN)...")
res_ddqn = test_robustness(DDQN_model, winds, turbulences, num_test_eps)

# Side-by-side plotting for report
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# Plots results for each wind speed and turbulence
for i, w in enumerate(winds):
    ax1.plot(turbulences, res_std[i, :], marker='o', label=f'Wind={w}')
    ax2.plot(turbulences, res_ddqn[i, :], marker='s', label=f'Wind={w}')

ax1.set_title('Standard DQN Robustness')
ax1.set_xlabel('Turbulence Power')
ax1.set_ylabel(f'Avg Reward ({num_test_eps} eps)')
ax1.legend()
ax1.grid(True)

ax2.set_title('Double DQN Robustness')
ax2.set_xlabel('Turbulence Power')
ax2.legend()
ax2.grid(True)

plt.show()

## Visualisation


In [ ]:
# Makes the environment have a video renderer for DQN
env = gym.make("LunarLander-v3", render_mode="rgb_array")
env = rl.RenderFrame(env, "./video_dqn")

obs, info = env.reset(seed=42)

while True:
  action = epsilon_greedy_policy(DQN_model, 0.0, obs, env)
  obs, reward, terminated, truncated, info = env.step(action)

  if terminated or truncated:
    break

env.play()

# Makes the environment have a video renderer for DDQN
env = gym.make("LunarLander-v3", render_mode="rgb_array")
env = rl.RenderFrame(env, "./video_ddqn")

obs, info = env.reset(seed=42)

while True:
  action = epsilon_greedy_policy(DDQN_model, 0.0, obs, env)
  obs, reward, terminated, truncated, info = env.step(action)

  if terminated or truncated:
    break

env.play()